# OsteoClassAI v2 · 08 Modelo robusto para la demo (D, 224 px)

Corrida **D**: misma receta que A (ConvNeXt-Tiny, SORD + TTA, **224 px**, para que la página siga siendo rápida) con `--aug robust`, igual que C.

El modelo se elige con la validación de Chen y la prueba de Chen se evalúa una vez. Después se mide en Gornale junto con A y C (Gornale ya no es imparcial).

**Antes de ejecutar:** `Digital Knee X-ray Images.zip` debe seguir en `MyDrive/OsteoClassAI/datos_externos/`.

*Entorno de ejecución → Cambiar tipo → GPU (T4)*, y luego *Ejecutar todas* (~20 min).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm kagglehub imagehash

In [ ]:
import kagglehub, pathlib
path = pathlib.Path(kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity'))
root = next(p.parent for p in path.rglob('train') if p.is_dir())
!python -m osteoclass.data --root "$root" --out outputs/data > data.log && tail -n 3 data.log

## D · SORD + TTA, 224 px, aumentos de robustez

In [ ]:
!python -m osteoclass.train --root "$root" --splits outputs/data/splits.csv --model convnext_tiny --loss sord --tta --aug robust --epochs 30 --patience 8 --out outputs/runs/convnext_tiny_sord_robust
!mkdir -p /content/drive/MyDrive/OsteoClassAI/runs && cp -r outputs/runs/convnext_tiny_sord_robust /content/drive/MyDrive/OsteoClassAI/runs/

## Gornale (ya no imparcial): A, C y D

In [ ]:
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/Digital Knee X-ray Images.zip'
!mkdir -p /content/gornale && cd /content/gornale && unzip -q -o "$ZIP"
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd "/content/gornale/Digital Knee X-ray Images/Knee X-ray Images" && for f in *.rar; do unrar x -o+ -idq "$f"; done
groot = '/content/gornale/Digital Knee X-ray Images/Knee X-ray Images'
RUNS = '/content/drive/MyDrive/OsteoClassAI/runs'
!python -m osteoclass.external --root "$groot" --runs $RUNS/convnext_tiny_sord $RUNS/convnext_tiny_sord_288_robust $RUNS/convnext_tiny_sord_robust --exploratory --out outputs/external/gornale_demo
!mkdir -p /content/drive/MyDrive/OsteoClassAI/external && cp -r outputs/external/gornale_demo /content/drive/MyDrive/OsteoClassAI/external/